# 🔍 Validación: Antes y Después de la Limpieza

Este notebook te permite ver con tus propios ojos cómo cambian los datos
antes y después de aplicar cada función de limpieza.

**Cómo usarlo:**
1. Corre la celda de configuración (Celda 1)
2. Corre la celda de carga de datos (Celda 2)
3. Ejecuta la validación de la columna que te interese

---

## Celda 1 — Configuración y carga de datos

In [1]:
# Carga los datos crudos desde Bronze
storage_account = "stecommercedirty01"  # cambia por el tuyo
container = "bronze"

path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/customers/"
df_raw = spark.read.option("multiLine", True).json(path)

print(f"✅ Cargados {df_raw.count()} registros")
print(f"📋 Columnas: {df_raw.columns}")

✅ Cargados 217 registros
📋 Columnas: ['city', 'country', 'created_at', 'email', 'id', 'name', 'phone', 'registration_date', 'updated_at', 'ingestion_date']


---
## 📝 Cómo ver ANTES y DESPUÉS de una columna

La técnica es simple:
1. Seleccionas la columna original
2. Aplicas la función de limpieza
3. Muestras AMBAS columnas juntas para comparar

Usamos `.select()` para ver solo las columnas que importan.

### 🔤 Validación: columna `name`

In [23]:
from functools import reduce
from pyspark.sql.functions import col
from transformacion.cleaning.customer_cleaner import clean_text_column

# 1) Aplica TODA la limpieza que tengas hasta ahora
df_limpio = clean_text_column(df_raw, "name")
# Si ya tienes la limpieza de país, aplícala aquí también, por ejemplo:
# df_limpio = limpiar_customers(df_limpio, spark)

# 2) Descubre qué columnas existen y cuáles creó la limpieza
cols_originales = df_raw.columns
cols_nuevas = [c for c in df_limpio.columns if c not in cols_originales]
print("Columnas originales:", cols_originales)
print("Columnas nuevas creadas por la limpieza:", cols_nuevas)

# 3) Orden de visualización: cada original seguida de su versión limpia
orden = []
for c in cols_originales:
    orden.append(c)
    for sufijo in ("_clean", "_final"):
        if f"{c}{sufijo}" in df_limpio.columns:
            orden.append(f"{c}{sufijo}")
orden += [c for c in df_limpio.columns if c not in orden]  # cualquier otra nueva

# 4) TODAS las filas, todas las columnas, antes y después lado a lado
df_limpio.select(orden).show()

# 5) Solo las filas donde algo cambió, en cualquier columna
pares = [(c, f"{c}{s}") for c in cols_originales for s in ("_clean", "_final")
         if f"{c}{s}" in df_limpio.columns]

if pares:
    hubo_cambio = reduce(lambda a, b: a | b,
                         [~col(o).eqNullSafe(col(l)) for o, l in pares])
    df_cambiaron = df_limpio.filter(hubo_cambio)
    print(f"Filas con al menos un cambio: {df_cambiaron.count()} de {df_limpio.count()}")
    df_cambiaron.select(orden).show(30,truncate=False)

    # 6) Resumen por columna: cuántos valores cambiaron en cada una
    for o, l in pares:
        n = df_limpio.filter(~col(o).eqNullSafe(col(l))).count()
        print(f"{o} -> {l}: {n} valores cambiaron")

Columnas originales: ['city', 'country', 'created_at', 'email', 'id', 'name', 'phone', 'registration_date', 'updated_at', 'ingestion_date']
Columnas nuevas creadas por la limpieza: ['name_clean']
+-------------+--------------+----------+--------------------+---+-----------------+-----------------+------------+-----------------+----------+--------------+
|         city|       country|created_at|               email| id|             name|       name_clean|       phone|registration_date|updated_at|ingestion_date|
+-------------+--------------+----------+--------------------+---+-----------------+-----------------+------------+-----------------+----------+--------------+
|  Los Angeles|            US|      NULL|david  .!@#35@noe...|  1|      David   !@#|            David|+12350438287|       2022-07-31|      NULL|    2026-08-30|
|     new york|           USA|      NULL|a.anderson79@emai...|  2|       a Anderson|       A Anderson|           +|       2023-09-23|      NULL|    2026-08-30|
|San

In [24]:
# ─── Detectar problemas: ¿Cuántos quedaron como UNKNOWN? ──────────────────────
from pyspark.sql import functions as F

total = df_name_check.count()
unknown_count = df_name_check.filter(F.col("name_clean") == "UNKNOWN").count()
changed_count = df_name_check.filter(F.col("name") != F.col("name_clean")).count()

print(f"📊 Total de registros: {total:,}")
print(f"✏️  Registros que cambiaron: {changed_count:,} ({changed_count/total*100:.1f}%)")
print(f"❓ Quedaron como UNKNOWN:  {unknown_count:,} ({unknown_count/total*100:.1f}%)")

# Ver cuáles quedaron como UNKNOWN para entender por qué
if unknown_count > 0:
    print("\n⚠️ Registros que quedaron UNKNOWN:")
    df_name_check.filter(F.col("name_clean") == "UNKNOWN") \
        .select("name", "name_clean") \
        .display()

📊 Total de registros: 217
✏️  Registros que cambiaron: 201 (92.6%)
❓ Quedaron como UNKNOWN:  0 (0.0%)


---
### 📧 Validación: columna `email`

In [22]:
# ─── ANTES y DESPUÉS de limpiar 'email' ──────────────────────────────────────
from pyspark.sql import functions as F  
from transformacion.cleaning.customer_cleaner import clean_email_column

df_email_check = clean_email_column(df_raw)

# Ver todos: original vs limpio
df_email_check.select("email", "email_clean").show(30,truncate=False)

# Ver solo los que quedaron como UNKNOWN (emails inválidos)
print("\n❌ Emails que NO pasaron validación:")
df_email_check.filter(
    F.col("email_clean") == "UNKNOWN"
).select("email", "email_clean").show(30,truncate=False)

+------------------------------+------------------------------+
|email                         |email_clean                   |
+------------------------------+------------------------------+
|david  .!@#35@noemail         |UNKNOWN                       |
|a.anderson79@email.com        |a.anderson79@email.com        |
|chris.williams47@123          |UNKNOWN                       |
|null.123                      |UNKNOWN                       |
|user.taylor44@outlook.com     |user.taylor44@outlook.com     |
|admin.johnson85@company       |UNKNOWN                       |
|x@fake.com.com.fake.com       |x@fake.com.com.fake.com       |
|pedro. brown35@yahoo.com      |UNKNOWN                       |
|carlos.smith                  |UNKNOWN                       |
|!@#.taylor96@email.com        |UNKNOWN                       |
|   .smith33@gmail.com         |.smith33@gmail.com            |
|anna.taylor51@invalid         |UNKNOWN                       |
|  ab@gmail.com                |ab@gmail

---
### 🌍 Validación: columna `country`

In [27]:
# ─── ANTES y DESPUÉS de normalizar 'country' ─────────────────────────────────
from transformacion.cleaning.country_normalizer import normalize_country

df_country_check = normalize_country(df_raw, spark)

# Ver todos los valores únicos: original vs estandarizado
# .distinct() evita mostrar miles de filas repetidas
print("🌍 Valores únicos de country: original → limpio")
df_country_check.select("country", "country_clean") \
    .distinct() \
    .orderBy("country") \
    .show(30,truncate=False)

# Resumen de resultados
print("\n📊 Distribución de resultados:")
df_country_check.groupBy("country_clean") \
    .count() \
    .orderBy(F.col("count").desc()) \
    .show(30,truncate=False)

# Ver los que necesitan revisión manual
revisar = df_country_check.filter(F.col("country_clean") == "REVISAR_MANUAL")
if revisar.count() > 0:
    print("\n⚠️ Países que necesitan revisión manual:")
    revisar.select("country").distinct().show(30,truncate=False)

🌍 Valores únicos de country: original → limpio
+--------------+--------------+
|country       |country_clean |
+--------------+--------------+
|NULL          |UNKNOWN       |
|              |UNKNOWN       |
|  UK          |United Kingdom|
|Canada        |Canada        |
|Germany       |Germany       |
|Mexico        |Mexico        |
|U.S.A.        |United States |
|US            |United States |
|USA           |United States |
|United Kingdom|United Kingdom|
|United States |United States |
|Usa           |United States |
|canada        |Canada        |
|mexico        |Mexico        |
|usa           |United States |
+--------------+--------------+


📊 Distribución de resultados:
+--------------+-----+
|country_clean |count|
+--------------+-----+
|United States |78   |
|Canada        |45   |
|UNKNOWN       |33   |
|Mexico        |26   |
|United Kingdom|25   |
|Germany       |10   |
+--------------+-----+



---
### 📅 Validación: columnas de fecha

In [ ]:
# ─── ANTES y DESPUÉS de formatear fechas ─────────────────────────────────────
from transformacion.cleaning.customer_cleaner import clean_date_columns

DATE_COLS = ["created_at", "updated_at", "registration_date"]
DATE_COLS_CLEAN = [f"{c}_clean" for c in DATE_COLS]  # created_at_clean, ...

# Guardamos los originales ANTES de limpiar
df_antes = df_raw.select(
    "id",
    *DATE_COLS  # * desempaqueta la lista como argumentos
).limit(20)

# Aplicamos la limpieza (crea columnas _clean, NO reemplaza las originales)
df_fechas_clean = clean_date_columns(df_raw, DATE_COLS)

# DESPUÉS: originales + versiones _clean para comparar lado a lado
df_despues = df_fechas_clean.select("id", *DATE_COLS, *DATE_COLS_CLEAN).limit(20)

print("📅 ANTES (raw):")
df_antes.show(20,truncate=False)

print("\n📅 DESPUÉS (originales + _clean):")
df_despues.show(20,truncate=False)

# Detectar fechas que NO pudieron parsearse (quedaron null en _clean)
for col_name in DATE_COLS_CLEAN:
    nulls_after = df_fechas_clean.filter(F.col(col_name).isNull()).count()
    if nulls_after > 0:
        print(f"\n⚠️ '{col_name}': {nulls_after} fechas no pudieron parsearse")
        df_fechas_clean.filter(F.col(col_name).isNull()) \
            .select("id", col_name) \
            .show(20,truncate=False)

---
## 🔄 Validación COMPLETA: pipeline entero

In [0]:
# ─── Correr el pipeline completo y ver el resultado final ─────────────────────
from transformacion.pipeline_transformacion import run_transformacion

dataframes = {"customers": df_raw}
dataframes_clean = run_transformacion(dataframes, spark)
df_final = dataframes_clean["customers"]

# Ver schema del resultado (tipos de datos después del pipeline)
print("📋 Schema FINAL (tipos de datos correctos):")
df_final.printSchema()

# Ver muestra del resultado
print("\n👀 Muestra del resultado limpio:")
df_final.display()

# Diagnóstico de nulos en el resultado
print("\n🚨 Nulos en el resultado final:")
df_final.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in df_final.columns
]).display()